# 2장. 특성 공학

모델은 무엇을 보고 배우는가.

**준비:** 국토교통부 실거래가 공개시스템에서 받은 서울 아파트 매매 CSV 네 개(2023~2026년)를 왼쪽 파일 창에 올린 뒤 위에서부터 차례로 실행한다. 『데이터 분석 완전 정복』 3장에서 쓴 파일 그대로다.

## 2.1 숫자 열만으로 가격 예측하기

In [ ]:
import os, re
import numpy as np
import pandas as pd

pd.set_option("display.unicode.east_asian_width", True)

def period(path):
    """파일 머리말에서 계약일자 범위를 읽는다. 실거래가 파일이 아니면 빈 리스트."""
    with open(path, encoding="cp949", errors="ignore") as f:
        head = "".join(f.readline() for _ in range(15))
    return re.findall(r"계약일자 : (\d{4}-\d{2}-\d{2}) ~ (\d{4}-\d{2}-\d{2})", head)

files = sorted(f for f in os.listdir(".") if f.endswith(".csv") and period(f))
raw = pd.concat([pd.read_csv(f, encoding="cp949", skiprows=15, thousands=",",
                             dtype={"번지": str, "본번": str, "부번": str, "동": str}) for f in files],
                ignore_index=True)

v = raw[(raw["해제사유발생일"] == "-") & (raw["거래유형"] != "직거래")].copy()
v = v.rename(columns={"전용면적(㎡)": "면적"})
v["억"] = v["거래금액(만원)"] / 1e4
v["날짜"] = pd.to_datetime(v["계약년월"].astype(str) + v["계약일"].astype(str).str.zfill(2), format="%Y%m%d")
v["구"] = v["시군구"].str.split().str[1]
v["동네"] = v["시군구"].str.split().str[2]
v["집"] = v["시군구"] + " " + v["번지"] + " " + v["단지명"] + " " + v["면적"].round().astype(int).astype(str)
v = v.sort_values("날짜").reset_index(drop=True)
print(len(files), raw.shape, len(v))

In [ ]:
train = v[v["날짜"].dt.year <= 2024].copy()
test = v[v["날짜"].dt.year == 2025].copy()

print(len(train), len(test))
print(train[["날짜", "구", "동네", "단지명", "면적", "층", "건축년도", "억"]].iloc[[0, 40000]].T.to_string())

In [ ]:
def rmse(actual, predicted):
    miss = np.asarray(actual, dtype=float) - np.asarray(predicted, dtype=float)
    return round(float(np.sqrt((miss ** 2).mean())), 2)

def off(actual, predicted):
    """답이 실제 값에서 몇 %나 벗어났는지를 거래마다 구해, 그 한가운데 값을 돌려준다."""
    ratio = np.asarray(predicted, dtype=float) / np.asarray(actual, dtype=float) - 1
    return round(float(np.median(np.abs(ratio)) * 100), 1)

BOARD = []

def score(name, make, log=False):
    """make가 만든 열로 2024년까지의 가격에 식을 맞추고, 두 자료에서 오차를 잰다."""
    Xtr, Xte = make(train), make(test)
    y = np.log(train["억"].to_numpy()) if log else train["억"].to_numpy()
    coef = np.linalg.lstsq(Xtr, y, rcond=None)[0]
    ptr, pte = Xtr @ coef, Xte @ coef
    if log:
        ptr, pte = np.exp(ptr), np.exp(pte)
    row = (name, Xtr.shape[1], rmse(train["억"], ptr), off(train["억"], ptr), rmse(test["억"], pte), off(test["억"], pte))
    BOARD.append(row)
    print(f"{name}: 열 {row[1]}개 | 2023~24년 {row[2]}억 {row[3]}% | 2025년 {row[4]}억 {row[5]}%")
    return coef, pte

def ones(df):
    return np.ones((len(df), 1))

score("평균만 답하기", ones);

In [ ]:
coef, _ = score("면적", lambda df: np.column_stack([ones(df), df["면적"]]))
print(coef.round(3).tolist())

In [ ]:
coef, _ = score("면적, 층, 건축년도", lambda df: np.column_stack([ones(df), df["면적"], df["층"], df["건축년도"]]))
print(coef.round(3).tolist())

## 2.2 범주형 변수 인코딩: 번호 매기기와 원-핫 인코딩

In [ ]:
GU = sorted(v["구"].unique())
number = {g: i + 1 for i, g in enumerate(GU)}
print(list(number.items())[:4], number["서초구"], number["도봉구"])

coef, _ = score("구를 번호로", lambda df: np.column_stack([ones(df), df["면적"], df["층"], df["건축년도"], df["구"].map(number)]))
print(coef.round(3).tolist())

In [ ]:
gu_median = train.groupby("구")["억"].median()
r = float(np.corrcoef([number[g] for g in GU], gu_median[GU])[0, 1])
print(gu_median[["강남구", "강동구", "강북구", "강서구", "도봉구", "서초구"]].round(2).to_dict())
print(round(r, 3))

In [ ]:
def onehot(s, cats):
    """값마다 열을 하나씩 두고, 그 값인 줄에만 1을 적는다."""
    return (s.to_numpy()[:, None] == np.asarray(cats)[None, :]).astype(float)

print(onehot(train["구"].iloc[:3], GU)[:, :6], train["구"].iloc[:3].tolist())

coef, _ = score("구를 열 스물다섯 개로", lambda df: np.column_stack([df["면적"], df["층"], df["건축년도"], onehot(df["구"], GU)]))
by_gu = pd.Series(coef[3:], index=GU)
print((by_gu - by_gu["도봉구"]).round(2).sort_values().iloc[[0, 1, 2, -3, -2, -1]].to_string())

In [ ]:
with_const = np.column_stack([ones(train), train["면적"], train["층"], train["건축년도"], onehot(train["구"], GU)])
no_const = with_const[:, 1:]
print(with_const.shape[1], np.linalg.matrix_rank(with_const), no_const.shape[1], np.linalg.matrix_rank(no_const))

In [ ]:
!pip -q install koreanize-matplotlib

In [ ]:
import matplotlib.pyplot as plt
import koreanize_matplotlib
from matplotlib.colors import to_rgba

NAVY, BRICK, TEAL, GRAY = "#16324f", "#b5532a", "#2a7f73", "#9aa1ab"


def frame(ax, title=None, color=NAVY, grid="y"):
    """그래프마다 반복하는 꾸밈: 제목, 보조선, 테두리 정리."""
    if title:
        ax.set_title(title, fontsize=9, loc="left", color=color, weight="bold")
    ax.grid(axis=grid, color="#e3e7ec", lw=0.6)
    ax.set_axisbelow(True)
    ax.tick_params(labelsize=8)
    ax.spines[["top", "right"]].set_visible(False)

In [ ]:
# 그림 2-2
nums = np.array([number[g] for g in GU])
fig, ax = plt.subplots(figsize=(7, 3.9))
ax.scatter(nums, gu_median[GU], s=28, color=to_rgba(NAVY, 0.25), edgecolor=NAVY, lw=0.8)
slope, base = np.polyfit(nums, gu_median[GU], 1)
ax.plot([1, 25], [base + slope, base + slope * 25], color=BRICK, lw=1.4, ls="--")
ax.text(25.2, base + slope * 25, f"번호와 가격의\n상관계수 {r:.2f}", color=BRICK, fontsize=8, va="center")
for g in ["강남구", "서초구", "송파구", "용산구", "도봉구", "강북구", "금천구"]:
    ax.annotate(g, (number[g], gu_median[g]), (number[g] + 0.35, gu_median[g] + 0.5), fontsize=7.5, color=NAVY)
ax.set_xticks(range(1, 26))
ax.set_xlim(0, 29.5)
ax.set_xlabel("가나다순으로 매긴 구의 번호", fontsize=8)
ax.set_ylabel("2023~24년 거래 가격의 중앙값(억 원)", fontsize=8)
frame(ax)
fig.tight_layout()
fig.savefig("fig_2_2.png", dpi=300)
plt.show()

In [ ]:
# 그림 2-4
gap = (by_gu - by_gu["도봉구"]).sort_values()
fig, ax = plt.subplots(figsize=(7, 5.6))
face = [to_rgba(BRICK, 0.55) if v > 10 else to_rgba(NAVY, 0.22) for v in gap]
edge = [BRICK if v > 10 else NAVY for v in gap]
ax.barh(gap.index, gap, color=face, edgecolor=edge, lw=0.9, height=0.66)
for i, v in enumerate(gap):
    ax.text(v + 0.2 if v >= 0 else v - 0.2, i, f"{v:+.1f}", va="center", ha="left" if v >= 0 else "right", fontsize=7.5)
ax.axvline(0, color=GRAY, lw=0.8)
ax.text(0.15, -1.25, "기준: 도봉구 = 0", fontsize=7.5, color=GRAY)
ax.set_xlim(-1.8, 17.5)
ax.set_xlabel("면적, 층, 건축년도가 같을 때 도봉구와의 가격 차이(억 원)", fontsize=8)
frame(ax, grid="x")
fig.tight_layout()
fig.savefig("fig_2_4.png", dpi=300)
plt.show()

## 2.3 로그 변환과 구간화

In [ ]:
price, logp = train["억"], np.log(train["억"])
print(round(float(price.mean()), 2), round(float(price.median()), 2), round(float(price.skew()), 2))
print(round(float(logp.mean()), 3), round(float(logp.median()), 3), round(float(logp.skew()), 2))
print(round(float(np.exp(logp.mean())), 2))

In [ ]:
# 그림 2-5
fig, axes = plt.subplots(2, 1, figsize=(7, 5.6))
axes[0].hist(price.clip(upper=60), bins=120, color=to_rgba(NAVY, 0.22), edgecolor=NAVY, lw=0.3)
axes[0].set_ylim(0, 4900)
for val, name, col in [(price.mean(), "평균", BRICK), (price.median(), "중앙값", TEAL)]:
    axes[0].axvline(val, color=col, lw=1.1, ls="--")
axes[0].text(price.mean() + 0.6, axes[0].get_ylim()[1] * 0.9, f"평균 {price.mean():.2f}억", color=BRICK, fontsize=8, bbox={"fc": "white", "ec": "none", "alpha": 0.85, "pad": 1.5})
axes[0].text(price.median() - 0.6, axes[0].get_ylim()[1] * 0.9, f"중앙값 {price.median():.2f}억", color=TEAL, fontsize=8, ha="right", bbox={"fc": "white", "ec": "none", "alpha": 0.85, "pad": 1.5})
axes[0].set_xlabel("거래 가격(억 원, 60억 초과는 60억으로 표시)", fontsize=8)
frame(axes[0], "가격 그대로: 오른쪽으로 긴 꼬리가 있다")
axes[1].hist(logp, bins=120, color=to_rgba(NAVY, 0.22), edgecolor=NAVY, lw=0.3)
for val, col in [(logp.mean(), BRICK), (logp.median(), TEAL)]:
    axes[1].axvline(val, color=col, lw=1.1, ls="--")
ticks = [1, 2, 5, 10, 20, 50, 100]
axes[1].set_xticks(np.log(ticks), [f"{t}억" for t in ticks])
axes[1].set_xlabel("거래 가격의 로그(눈금은 원래 가격으로 표시)", fontsize=8)
frame(axes[1], "가격의 로그: 좌우가 비슷해지고 평균과 중앙값이 가까워진다")
for ax in axes:
    ax.set_ylabel("거래 건수", fontsize=8)
fig.tight_layout()
fig.savefig("fig_2_5.png", dpi=300)
plt.show()

In [ ]:
score("가격에 로그", lambda df: np.column_stack([df["면적"], df["층"], df["건축년도"], onehot(df["구"], GU)]), log=True)
coef, _ = score("면적에도 로그", lambda df: np.column_stack([np.log(df["면적"]), df["층"], df["건축년도"], onehot(df["구"], GU)]), log=True)
print(round(float(coef[0]), 3), round(float(coef[1]), 4), round(float(coef[2]), 4))

In [ ]:
def X_log(df):
    return np.column_stack([np.log(df["면적"]), df["층"], df["건축년도"], onehot(df["구"], GU)])

b = np.linalg.lstsq(X_log(train), np.log(train["억"].to_numpy()), rcond=None)[0]
resid = np.log(train["억"].to_numpy()) - X_log(train) @ b
factor = float(np.exp(resid).mean())
plain = np.exp(X_log(train) @ b)
print(round(float(resid.std()), 3), round(factor, 3), round(float(np.exp(resid.var() / 2)), 3))
print(round(float(train["억"].mean()), 2), round(float(plain.mean()), 2), round(float((plain * factor).mean()), 2))
test_plain = np.exp(X_log(test) @ b)
print(rmse(test["억"], test_plain), off(test["억"], test_plain), rmse(test["억"], test_plain * factor), off(test["억"], test_plain * factor))

In [ ]:
train["연식"] = train["날짜"].dt.year - train["건축년도"]
test["연식"] = test["날짜"].dt.year - test["건축년도"]
train["㎡당"] = train["억"] * 1e4 / train["면적"]              # 만 원

decade = train["연식"].clip(0, 59) // 10 * 10
print(train.groupby(decade)["㎡당"].agg(["size", "median"]).round(0).astype(int).to_string())

In [ ]:
# 그림 2-6
by_age = train[train["연식"].between(0, 50)].groupby("연식")["㎡당"].median()
slope, base = np.polyfit(train["연식"], train["㎡당"], 1)
fig, ax = plt.subplots(figsize=(7, 3.9))
ax.bar(by_age.index, by_age.values, color=to_rgba(NAVY, 0.22), edgecolor=NAVY, lw=0.6, width=0.8)
ax.plot([0, 50], [base, base + slope * 50], color=BRICK, lw=1.6)
ax.text(1, base + 330, f"직선으로 나타내면 1년에 {slope:+.1f}만 원", color=BRICK, fontsize=8, bbox={"fc": "white", "ec": "none", "alpha": 0.85, "pad": 1.5})
low = int(by_age.loc[15:35].idxmin())
ax.annotate(f"가장 낮은 곳\n{low}년, {by_age[low]:,.0f}만 원", (low, by_age[low]), (low - 3, by_age[low] + 900), fontsize=8, color=TEAL, ha="center", arrowprops={"arrowstyle": "->", "color": TEAL, "lw": 0.8})
ax.annotate("30년을 넘으면 다시 오른다\n(재건축 기대)", (41, by_age[41]), (30.5, by_age.max() * 0.93), fontsize=8, color=NAVY, arrowprops={"arrowstyle": "->", "color": NAVY, "lw": 0.8})
for d in range(10, 50, 10):
    ax.axvline(d - 0.5, color=GRAY, lw=0.6, ls=":")
ax.set_xlabel("연식 = 계약한 해 - 지은 해(년). 점선은 10년 단위 구간의 경계", fontsize=8)
ax.set_ylabel("㎡당 가격의 중앙값(만 원)", fontsize=8)
frame(ax)
fig.tight_layout()
fig.savefig("fig_2_6.png", dpi=300)
plt.show()

In [ ]:
AGES = [0, 10, 20, 30, 40, 50]

def basic(df):
    """면적의 로그, 층, 연식을 십 년 단위로 자른 열 여섯 개."""
    return np.column_stack([np.log(df["면적"]), df["층"], onehot(df["연식"].clip(0, 59) // 10 * 10, AGES)])

coef, _ = score("연식을 십 년 단위로", lambda df: np.column_stack([basic(df), onehot(df["구"], GU)[:, 1:]]), log=True)
print((np.exp(coef[2:8] - coef[4]) * 100 - 100).round(1).tolist())

In [ ]:
DONG = sorted(train["동네"].unique())
print(len(DONG), int((~test["동네"].isin(DONG)).sum()))

score("구 대신 동으로", lambda df: np.column_stack([basic(df), onehot(df["동네"], DONG)[:, 1:]]), log=True);

## 2.4 목표 인코딩과 데이터 누수

In [ ]:
typical = train.groupby("집")["억"].median()
gu_typical = train.groupby("구")["억"].median()
print(len(typical), round(float(test["집"].isin(typical.index).mean()), 3))

def with_past(df, past):
    """past가 빈 줄은 구의 중앙값으로 메우고, 그 로그를 열로 붙인다."""
    filled = past.fillna(df["구"].map(gu_typical))
    return np.column_stack([basic(df), onehot(df["구"], GU)[:, 1:], np.log(filled)])

train["지난가"] = train["집"].map(typical)
test["지난가"] = test["집"].map(typical)
coef, guess = score("같은 집의 중앙값", lambda df: with_past(df, df["지난가"]), log=True)
known = test["지난가"].notna().to_numpy()
print(round(float(coef[-1]), 3))
print(int(known.sum()), off(test["억"][known], guess[known]), int((~known).sum()), off(test["억"][~known], guess[~known]))

In [ ]:
def others(s):
    """같은 집의 거래들 가운데 자기 자신을 뺀 나머지의 중앙값. 거래가 하나뿐이면 빈 값."""
    a = s.to_numpy()
    if len(a) == 1:
        return pd.Series([np.nan], index=s.index)
    return pd.Series([np.median(np.delete(a, i)) for i in range(len(a))], index=s.index)

train["지난가"] = train.groupby("집")["억"].transform(others)
print(round(float(train["지난가"].isna().mean()), 3), round(float(test["지난가"].isna().mean()), 3))

coef, _ = score("자기 거래를 뺀 중앙값", lambda df: with_past(df, df["지난가"]), log=True)
print(round(float(coef[-1]), 3))

In [ ]:
dong_typical = train.groupby("동네")["억"].median()

def with_past2(df):
    """빈 줄은 동의 중앙값, 그것도 없으면 구의 중앙값으로 메우고, 메웠다는 표시를 열로 남긴다."""
    past = df["지난가"]
    filled = past.fillna(df["동네"].map(dong_typical)).fillna(df["구"].map(gu_typical))
    return np.column_stack([basic(df), onehot(df["구"], GU)[:, 1:], np.log(filled), past.isna().astype(float)])

coef, guess = score("빈칸을 동으로 메우고 표시", with_past2, log=True)
print(int(known.sum()), off(test["억"][known], guess[known]), int((~known).sum()), off(test["억"][~known], guess[~known]))

In [ ]:
lean = pd.Series(test["억"].to_numpy() / guess - 1, index=test["날짜"].to_numpy())
print((lean.groupby(lean.index.quarter).median() * 100).round(1).to_string())

In [ ]:
board = pd.DataFrame(BOARD, columns=["보여 준 것", "열", "훈련 억", "훈련 %", "시험 억", "시험 %"]).set_index("보여 준 것")
print(board.to_string())

In [ ]:
# 그림 2-8
names = ["같은 집의 중앙값", "자기 거래를 뺀 중앙값", "빈칸을 동으로 메우고 표시"]
labels = ["자기 거래를 포함한 중앙값\n(누수 있음)", "자기 거래를 뺀 중앙값", "빈칸을 동의 중앙값으로\n메우고 표시 열 추가"]
fig, ax = plt.subplots(figsize=(7, 3.8))
pos = np.arange(3)
tr, te = board.loc[names, "훈련 %"], board.loc[names, "시험 %"]
ax.bar(pos - 0.19, tr, width=0.36, color=to_rgba(NAVY, 0.22), edgecolor=NAVY, lw=0.9, label="훈련 자료(2023~24년)")
ax.bar(pos + 0.19, te, width=0.36, color=to_rgba(BRICK, 0.55), edgecolor=BRICK, lw=0.9, label="시험 자료(2025년)")
for i in range(3):
    ax.text(i - 0.19, tr.iloc[i] + 0.25, f"{tr.iloc[i]}%", ha="center", fontsize=8, color=NAVY)
    ax.text(i + 0.19, te.iloc[i] + 0.25, f"{te.iloc[i]}%", ha="center", fontsize=8, color=BRICK)
    ax.text(i, 13.6, f"시험 ÷ 훈련 = {te.iloc[i] / tr.iloc[i]:.1f}배", ha="center", fontsize=8, color=TEAL)
ax.set_xticks(pos, labels)
ax.set_ylim(0, 15)
ax.set_ylabel("오차율의 중앙값(%)", fontsize=8)
ax.legend(frameon=False, fontsize=8, loc="upper left", bbox_to_anchor=(0.0, 0.88))
frame(ax)
fig.tight_layout()
fig.savefig("fig_2_8.png", dpi=300)
plt.show()

In [ ]:
# 그림 2-9
by_q = lean.groupby(lean.index.quarter).median() * 100
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.bar([f"2025년 {q}분기" for q in by_q.index], by_q, width=0.5, color=to_rgba(BRICK, 0.55), edgecolor=BRICK, lw=0.9)
for i, val in enumerate(by_q):
    ax.text(i, val + 0.3, f"+{val:.1f}%", ha="center", fontsize=8.5, color=BRICK)
ax.axhline(0, color=GRAY, lw=0.8)
ax.set_ylim(0, 17)
ax.set_ylabel("실제 가격이 식의 답보다 높은 정도(%)", fontsize=8)
frame(ax)
fig.tight_layout()
fig.savefig("fig_2_9.png", dpi=300)
plt.show()

In [ ]:
# 그림 2-10
show = board.drop(index=["가격에 로그", "같은 집의 중앙값", "자기 거래를 뺀 중앙값"])
fig, ax = plt.subplots(figsize=(7, 4.3))
y = np.arange(len(show))[::-1]
face = [to_rgba(GRAY, 0.35)] + [to_rgba(NAVY, 0.22)] * (len(show) - 2) + [to_rgba(BRICK, 0.55)]
edge = [GRAY] + [NAVY] * (len(show) - 2) + [BRICK]
ax.barh(y, show["시험 %"], color=face, edgecolor=edge, lw=0.9, height=0.62)
prev = None
for yi, val in zip(y, show["시험 %"]):
    drop = "" if prev is None else f"  ({val - prev:+.1f}%p)"
    ax.text(val + 0.5, yi, f"{val}%{drop}", va="center", fontsize=8)
    prev = val
ax.set_yticks(y)
ax.set_yticklabels([f"{n}  [열 {c}개]" for n, c in zip(show.index, show["열"])], fontsize=8)
ax.set_xlim(0, 47)
ax.set_xlabel("2025년 거래에서 오차율의 중앙값(%). 괄호 안은 바로 윗줄과의 차이", fontsize=8)
frame(ax, grid="x")
fig.tight_layout()
fig.savefig("fig_2_10.png", dpi=300)
plt.show()